# Homework 5: Monitoring

In [2]:
import urllib.request

# Define the exact 2026 monitoring cohort directory URL
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring"

# Download rag_helper.py
urllib.request.urlretrieve(f"{PREFIX}/rag_helper.py", "rag_helper.py")
print("✅ rag_helper.py downloaded successfully!")

# Download starter.py
urllib.request.urlretrieve(f"{PREFIX}/starter.py", "starter.py")
print("✅ starter.py downloaded successfully!")

✅ rag_helper.py downloaded successfully!
✅ starter.py downloaded successfully!


In [1]:
print(123)

123


In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by using a `while True` loop and checking whether the model returned any `function_call` items.

- It sends the current `messages` to the model.
- If the response includes a `function_call`, the code runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`.
- If the response has no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is simple: **no function calls in the latest response means the agent is done**.


In [6]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

### Q1. First trace

Wrap the rag() method so each call produces a span. The simplest way is to create a RAGTraced subclass of RAGBase that wraps rag(), search(), and llm() each in their own span.

Run this query:

How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary. Count the spans in the console output - each one is a separate ReadableSpan entry. How many spans does the trace produce?

1
3
5
7

In [ ]:
class RAGTraced(RAGBase):
    
    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            return super().search(query, num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            
            # Extract usage stats from your specific client response format
            usage = response.usage 
            
            # Set the metrics as attributes on the current active span
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            
            return response

### Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you want to them with set_attribute. We already use spans to record how long each step takes. Now we'll add the metrics we care about: tokens and cost.

Read the token usage from the LLM response (the llm() method in the starter already returns the raw response object) and set them as attributes on the llm span:

span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
And since we know both input and output tokens, we can also compute the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

700
7000
70000
700000
These numbers vary between runs. Pick the closest option.

### Q3. Span timing

Each span automatically records its duration. Look at the console output from Q1 and find the durations for the search span and the llm span.

For a typical query, roughly how long does the LLM call take?

Under 100ms
100-500ms
500-2000ms
Over 2000ms
The first call can be slower (cold start). Pick the range you see most often.

In [26]:
from starter import rag
# Assuming you defined RAGTraced in your file or imported it
# class RAGTraced(RAGBase): ...

# 1. Re-initialize using your traced subclass, passing the components from the starter object
traced_rag = RAGTraced(index=rag.index, llm_client=rag.llm_client)

# 2. Run the query on the traced instance
query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x529e695ab53c821eca955994e0cb2f8f",
        "span_id": "0x52e2640f2780076b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xaf78386761e0a2f1",
    "start_time": "2026-07-20T16:08:25.234374Z",
    "end_time": "2026-07-20T16:08:25.243230Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "a943e9a5-932f-4e8a-a293-8d195101bc0a",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x529e695ab53c821eca955994e0cb2f8f",
        "span_id": "0x1926a4f2e80e8320",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xaf78386761e0a2f1",
    "start_time": "2026-07-20T16:08:25.274645Z",
    "end_time": "2026-07-20T16:08:27.100087Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 89
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "a943e9a5-932f-4e8a-a293-8d195101bc0a",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
        "trace_id": "0x529e695ab53c821eca955994e0cb2f8f",
        "span_id": "0xaf78386761e0a2f

### Q4. Saving traces to SQLite

Right now the spans are printed to the terminal and then gone. We don't save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there. But we need to create a custom exporter. Instead of printing the spans, it will save them to the database.

OTel calls the exporter through the same span processor we already use, we just swap the destination.

Now we will create a custom exporter that saves each finished span to a SQLite database. The exporter extends SpanExporter. It has the following methods:

export method that receives a list of ReadableSpan objects
shutdown and force_flush methods

Re-run the query from Q1. Which span names appear in the spans table?

Only rag
rag and llm
rag, search, and llm
search, llm, and judge

In [17]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [18]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [20]:
import sqlite3

# Connect to the database file
conn = sqlite3.connect("traces.db")
cursor = conn.cursor()

# Fetch all rows from the spans table
cursor.execute("SELECT * FROM spans")
rows = cursor.fetchall()

# Print the columns neatly
print(f"{'Name':<10} | {'Start Time':<20} | {'End Time':<20} | {'In Tokens':<10} | {'Out Tokens':<10} | {'Cost':<5}")
print("-" * 85)

for row in rows:
    # row looks like: (name, start_time, end_time, input_tokens, output_tokens, cost)
    print(f"{str(row[0]):<10} | {str(row[1]):<20} | {str(row[2]):<20} | {str(row[3]):<10} | {str(row[4]):<10} | {str(row[5]):<5}")

conn.close()

Name       | Start Time           | End Time             | In Tokens  | Out Tokens | Cost 
-------------------------------------------------------------------------------------
search     | 1784562957379989000  | 1784562957578764200  | None       | None       | None 
llm        | 1784562957622187400  | 1784562960911684000  | 7111       | 126        | None 
rag        | 1784562957378302100  | 1784562960933927300  | None       | None       | None 


### Q5. Querying trace data
The traces are now in SQLite. Run one more query through the traced RAG, then query the database.

The rag span wraps everything, so its duration includes both search and llm. To see where time actually goes, exclude the rag span and compare the children.

Using SQL (or pandas), compute the total duration for each span name excluding rag. Which span type takes the most total time?

search
llm
They're all about the same

In [22]:
# --- 1. RUN ONE MORE QUERY THROUGH THE TRACED RAG ---

print("Running one more query...")
new_query = "What happens if the model never returns a stop signal?"
answer = traced_rag.rag(new_query) 
print(f"Answer: {answer}\n")


# --- 2. QUERY THE DATABASE ---



print("Querying the SQLite database for total durations...")

# Connect to the local database file where the exporter saves the spans
conn = sqlite3.connect("traces.db")
cursor = conn.cursor()

# Paste the exact SQL query requested
sql_query = """
SELECT 
    name, 
    SUM(end_time - start_time) AS total_duration
FROM spans
WHERE name != 'rag'
GROUP BY name;
"""

cursor.execute(sql_query)
results = cursor.fetchall()

# Print out your results neatly to compare 'search' and 'llm'
print("-" * 45)
print(f"{'Span Name':<12} | {'Total Duration':<20}")
print("-" * 45)
for row in results:
    print(f"{row[0]:<12} | {row[1]:<20}")
print("-" * 45)

conn.close()

Running one more query...
{
    "name": "search",
    "context": {
        "trace_id": "0xad8719593db515e98cd7bdc8268f7e83",
        "span_id": "0xa9e56bc3de7e396b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xd36a58db3809a8d6",
    "start_time": "2026-07-20T16:03:28.775245Z",
    "end_time": "2026-07-20T16:03:28.791354Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "a943e9a5-932f-4e8a-a293-8d195101bc0a",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xad8719593db515e98cd7bdc8268f7e83",
        "span_id": "0x3c0f8aed662acb96",
        "trace_state": "[]"
  

### Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you is how stable your system is. If the same query always produces the same number of input tokens, the context your RAG retrieves is consistent. If it varies a lot, something in the search may be unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls total in the database). Then compute the input tokens for each llm span.

How much do the input tokens vary across these 4 runs?

They're identical
Within 10% of each other
Within 50% of each other
They vary more than 50%

In [29]:

import pandas as pd

# 1. Connect to your database
conn = sqlite3.connect("traces.db")

# 2. Query the exact columns from your schema for 'llm' spans
query = "SELECT name, input_tokens, output_tokens FROM spans WHERE name = 'llm'"
df_spans = pd.read_sql_query(query, conn)

# 3. Print out your dataframe to see the 4 runs side-by-side
print(df_spans)

conn.close()

conn.close()

  name  input_tokens  output_tokens
0  llm          7111            126
1  llm          7494             50
2  llm          7111            106
3  llm          7111            108
4  llm          7111             98
5  llm          7111             89
